In [ ]:
!pip install -U "chromadb<0.6.0" "opentelemetry-sdk<1.27.0" "opentelemetry-api<1.27.0"
!pip install -U langchain_chroma

  Using cached chromadb-0.5.23-py3-none-any.whl.metadata (6.8 kB)
Using cached chromadb-0.5.23-py3-none-any.whl (628 kB)
  Attempting uninstall: chromadb
    Found existing installation: chromadb 1.5.9
    Uninstalling chromadb-1.5.9:
      Successfully uninstalled chromadb-1.5.9
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)


In [ ]:
!pip install langchain_google_genai
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from google.colab import userdata
from langchain_chroma import Chroma
from pydantic import BaseModel, Field

#STEP2 - Load api key
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

#STEP 3 - Initialize LLM and Embeddings Model
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# Correctly initialize the embeddings model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

class EmailClassificationOutput(BaseModel):
    urgency: str = Field(description="Classification of email urgency: High, Medium, or Low.")
    topic: str = Field(description="Main topic of discussion in the email (e.g., Accounts, Billing, Password reset).")
    response_text_category: str = Field(description="Category of response needed (e.g., Simple, Complex/Unresolved, needs customer support analyst).")
    follow_up_required: bool = Field(description="True if a follow-up action is required, False otherwise.")

class EscalationEmailOutput(BaseModel):
    recipient: str = Field(description="The email address of the internal customer support agent or team.")
    subject: str = Field(description="The subject line for the escalation email.")
    body: str = Field(description="The body of the escalation email, detailing the issue and reason for escalation.")

In [ ]:
class AnalysisOutput(BaseModel):
    analysis: str = Field(description="Detailed analysis of the email content and retrieved documents.")
    response_draft: str = Field(description="A draft response to the email based on the analysis and retrieved information.")
    follow_up_required: bool = Field(description="True if additional follow-up action is required after sending the drafted response, False otherwise.")

In [ ]:
from typing import TypedDict, Optional, List
from langchain_core.documents import Document

# Define the AgentState for managing the agent's workflow
class AgentState(TypedDict):
    email_content: str # Input email content for classification
    urgency: Optional[EmailClassificationOutput] # Using the previously defined structured output
    analysis: Optional[str]
    response_draft: Optional[str]
    follow_up_required: Optional[bool]
    retrieved_docs: Optional[List[Document]] # New field to store retrieved documents
    escalation_required: Optional[bool] # New field to indicate if escalation is needed
    escalation_email_draft: Optional[str] # New field for the drafted escalation email

In [ ]:
#!pip install -U langchain langchain-community langchain-text-splitters
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
kb_path = '/content/knowledge_base.rtf'
loader = TextLoader(kb_path)
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
vector_db = Chroma.from_documents(splitter.split_documents(docs), embeddings)
retriever = vector_db.as_retriever()

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# Define a system prompt for urgency identification
classify_system_prompt = """
You are an AI assistant tasked with analyzing email content and extracting structured information.
Your output MUST be a JSON object that adheres to the following schema:

```json
{schema}
```

Based on the email content, you need to classify the following:
1.  **Urgency**: Classify its urgency as one of the following categories:
    -   High: Requires immediate attention, critical, time-sensitive, potential for significant negative impact if delayed.
    -   Medium: Important, needs attention within a day or two, but not immediately critical.
    -   Low: Informational, non-urgent, can be addressed at convenience.

2.  **Topic**: Identify the main topic of discussion in the email (e.g., Accounts, Accounts, Billing, Password reset, Product Inquiry, Technical Issue, etc.).

3.  **Response Text Category**: Determine the category of response needed (e.g., Simple, Complex/Unresolved, needs customer support analyst, FAQ-answerable, etc.).

4.  **Follow-up Required**: Indicate whether a follow-up action is required (True/False).
"""

# Create a ChatPromptTemplate for urgency classification
classify_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", classify_system_prompt),
        ("user", "{email_content}")
    ]
)

# Create a chain with structured output
urgency_chain = classify_prompt_template | llm.with_structured_output(EmailClassificationOutput)

escalation_email_system_prompt = """
You are an AI assistant tasked with drafting an internal escalation email to a customer support agent.
Your output MUST be a JSON object that adheres to the following schema:

```json
{schema}
```

Based on the provided email content, its urgency classification, and any analysis:
1.  **Recipient**: Specify the email address of the internal customer support team or agent (e.g., 'support@example.com', 'noc@example.com').
2.  **Subject**: Create a clear and concise subject line for the escalation email, including the original email's subject and its urgency.
3.  **Body**: Draft a detailed body for the escalation email. This should include:
    -   A summary of the original email's problem.
    -   The determined urgency and topic.
    -   Any relevant analysis or context that might help the support agent.
    -   A clear request for the support agent to take over or provide guidance.

Original Email Content: {email_content}
Urgency Classification: {urgency_classification}
Analysis (if available): {analysis}
"""

escalation_email_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", escalation_email_system_prompt),
        ("user", "Draft an escalation email for this issue.")
    ]
)

escalation_email_chain = escalation_email_prompt_template | llm.with_structured_output(EscalationEmailOutput)

In [ ]:
analysis_system_prompt = """
You are an AI assistant tasked with analyzing an email and relevant retrieved documents to formulate a comprehensive analysis, draft a response, and determine if further follow-up is necessary.
Your output MUST be a JSON object that adheres to the following schema:

```json
{schema}
```

Based on the user's email and the provided context:
1.  **Analysis**: Provide a detailed analysis of the email's core issue, incorporating information from the retrieved documents.
2.  **Response Draft**: Generate a concise and helpful draft response to the user.
3.  **Follow-up Required**: Indicate whether any further action or follow-up is needed after this response (True/False).

Email Content: {email_content}

Retrieved Documents: {retrieved_docs}
"""

analysis_draft_prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", analysis_system_prompt),
        ("user", "Analyze the email and context to draft a response.")
    ]
)

analysis_draft_chain = analysis_draft_prompt_template | llm.with_structured_output(AnalysisOutput)